# SFT on a Base Model: Seeing the Difference

**Goal:** Load `Qwen3-4B` (base, not instruct), run inference to see raw next-token-prediction behaviour, fine-tune it on a chat/instruct dataset via SFT + LoRA, then compare outputs side-by-side.

**Key concept:** A *base* model has only seen raw text during pre-training. It has no notion of "user" and "assistant" roles. SFT (Supervised Fine-Tuning) teaches it to follow instructions by showing it many (instruction, response) pairs.

## 1. Install dependencies

In [1]:
%%capture
# unsloth: training-speed wrapper around HF transformers + PEFT
# trl   : provides SFTTrainer (the training loop)
# peft  : LoRA / adapter support
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch
    v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

## 2. Load the base model

We intentionally load **`Qwen3-4B`** (base) — not the `-Instruct` variant.  
The base model is a pure language model: given tokens, predict the next token.  
It will *complete* text, not *answer* questions.

In [2]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = "unsloth/Qwen3-4B",   # BASE model — no instruction tuning
    max_seq_length = 2048,                 # max tokens per sample during training
    load_in_4bit  = True,                  # 4-bit NF4 quantisation: cuts VRAM ~4x at minor quality cost
    load_in_8bit  = False,
    full_finetuning = False,               # we use LoRA (parameter-efficient), not full fine-tuning
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-03-17 12:27:06.231473: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773750426.450979      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773750426.514135      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773750427.026957      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773750427.027003      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773750427.027009      55 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Qwen3 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.55G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/237 [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

unsloth/qwen3-4b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


## 3. Baseline inference — before any training

Feed the model a bare instruction string (no chat template, no special tokens).  
Observe: the model continues the text rather than answering as an assistant.  
We capture the output in a variable so we can compare it later.

In [3]:
# Raw prompt — no ChatML wrapper, just plain text as the base model was trained on
raw_prompt = "What is the capital of France? Answer:"

inputs = tokenizer(raw_prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    out_ids = model.generate(
        **inputs,
        max_new_tokens = 80,
        temperature    = 0.7,
        top_p          = 0.9,
        do_sample      = True,
    )

# Decode only the newly generated tokens (skip the prompt)
before_response = tokenizer.decode(out_ids[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
print("=== BASE MODEL (before SFT) ===")
print(before_response)

=== BASE MODEL (before SFT) ===
 Paris. But why is Paris the capital of France? Because it is the largest city in France and the political, economic, and cultural center of the country. So the answer is Paris. Now, let's think about the question: What is the capital of Germany? The answer is Berlin. But why is Berlin the capital of Germany? Because it is the largest city in Germany and the political,


## 4. Attach LoRA adapters

LoRA (Low-Rank Adaptation) freezes all original weights and adds tiny trainable rank-decomposition matrices to selected layers.  
Only ~0.5–1 % of parameters are trained — much cheaper than full fine-tuning.

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,          # LoRA rank — higher = more capacity, more VRAM. 8–32 is typical
    lora_alpha     = 16,          # scaling factor; usually set equal to r
    lora_dropout   = 0.0,         # 0 is fine for most SFT jobs
    bias           = "none",      # don't train bias terms
    target_modules = [            # which weight matrices to inject LoRA into
        "q_proj", "k_proj", "v_proj", "o_proj",   # attention projections
        "gate_proj", "up_proj", "down_proj",       # MLP / FFN projections
    ],
    use_gradient_checkpointing = "unsloth",  # trades compute for memory — essential on T4
    random_state   = 42,
    use_rslora     = False,       # rank-stabilised LoRA variant — off for simplicity
)

Unsloth 2026.3.4 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


## 5. Load & prepare the instruction dataset

**FineTome-100k** is a curated, high-quality subset of ShareGPT conversations.  
Each row contains a `conversations` list of `{from, value}` dicts.  

We must format these into the **ChatML** token format that Qwen3 uses:  
```
<|im_start|>user
...<|im_end|>
<|im_start|>assistant
...<|im_end|>
```
The tokenizer's `apply_chat_template` handles this automatically.

In [5]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template, standardize_data_formats

# 1. Apply the Qwen3 ChatML template to the tokenizer
tokenizer = get_chat_template(tokenizer, chat_template="qwen3-instruct")

# 2. Load dataset
dataset = load_dataset("mlabonne/FineTome-100k", split="train")

# 3. Normalise the conversations field to a standard schema
#    (some datasets use 'human'/'gpt', others 'user'/'assistant' — this unifies them)
dataset = standardize_data_formats(dataset)

# 4. Render each conversation to a single formatted string stored in "text"
def apply_template(examples):
    return {
        "text": [
            tokenizer.apply_chat_template(
                convo,
                tokenize=False,
                add_generation_prompt=False,  # False during training — full turns already present
            )
            for convo in examples["conversations"]
        ]
    }

dataset = dataset.map(apply_template, batched=True)

# Peek at one example to verify the format
print(dataset[0]["text"][:500])

README.md:   0%|          | 0.00/982 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/117M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

Unsloth: Standardizing formats (num_proc=8):   0%|          | 0/100000 [00:00<?, ? examples/s]

Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

<|im_start|>user
Explain what boolean operators are, what they do, and provide examples of how they can be used in programming. Additionally, describe the concept of operator precedence and provide examples of how it affects the evaluation of boolean expressions. Discuss the difference between short-circuit evaluation and normal evaluation in boolean expressions and demonstrate their usage in code. 

Furthermore, add the requirement that the code must be written in a language that does not suppo


### What the training data looks like
One raw conversation vs the same conversation after the ChatML template is applied.

In [33]:
# Raw conversation (list of role/content dicts, as loaded from HuggingFace)
print("==== RAW ====")
print(dataset[1]["conversations"])

print("\n"+"=="*70)

print("\n==== AFTER CHAT TEMPLATE ====")
print(dataset[1]["text"])

==== RAW ====
[{'content': 'Explain how recursion works and provide a recursive function in Python that calculates the factorial of a given number.', 'role': 'user'}, {'content': "Recursion is a programming technique where a function calls itself to solve a problem. It breaks down a complex problem into smaller, more manageable subproblems until a base case is reached. The base case is a condition where the function does not call itself, but instead returns a specific value or performs a specific action.\n\nIn the case of calculating the factorial of a number, recursion can be used to break down the problem into simpler subproblems. The factorial of a non-negative integer n is the product of all positive integers less than or equal to n.\n\nHere is a recursive function in Python that calculates the factorial of a given number:\n\n```python\ndef factorial(n):\n    # Base case: factorial of 0 or 1 is 1\n    if n == 0 or n == 1:\n        return 1\n    # Recursive case: factorial of n is n

## 6. Configure and run SFT training

We train **only on assistant responses** (response masking).  
Loss on user turns is masked to -100 so the model never learns to imitate the user — only the assistant.

In [6]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field         = "text",     # column that contains formatted strings
        per_device_train_batch_size = 2,          # samples per GPU step
        gradient_accumulation_steps = 4,          # effective batch size = 2 * 4 = 8
        warmup_steps               = 5,           # linearly ramp LR for first N steps
        max_steps                  = 30,          # short run for demo; set num_train_epochs=1 for full training
        learning_rate              = 2e-4,        # higher LR is fine for short LoRA runs
        lr_scheduler_type          = "linear",    # LR decays linearly after warmup
        optim                      = "adamw_8bit",# 8-bit Adam saves VRAM with negligible quality loss
        weight_decay               = 0.01,
        logging_steps              = 1,
        seed                       = 42,
        output_dir                 = "./sft_output",
        report_to                  = "none",
    ),
)

# Mask loss on user turns — model only learns from assistant tokens
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",       # tokens to mask (user side)
    response_part    = "<|im_start|>assistant\n",  # tokens to learn from
)

Unsloth: Tokenizing ["text"] (num_proc=8):   0%|          | 0/100000 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


Map (num_proc=8):   0%|          | 0/100000 [00:00<?, ? examples/s]

Filter (num_proc=8):   0%|          | 0/100000 [00:00<?, ? examples/s]

Unsloth: Removed 94 out of 100000 samples from train_dataset where all labels were -100 (no response found after truncation). This prevents NaN loss during training.


In [7]:
# Optional: verify masking worked — labels for user turns should be -100
sample_input  = tokenizer.decode(trainer.train_dataset[0]["input_ids"])
sample_labels = tokenizer.decode(
    [tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[0]["labels"]]
).replace(tokenizer.pad_token, "[MASKED]")

print("Full input (first 400 chars):\n", sample_input[:400])
print("\nLabels — only assistant turns visible (first 400 chars):\n", sample_labels[:400])

Full input (first 400 chars):
 <|im_start|>user
Explain what boolean operators are, what they do, and provide examples of how they can be used in programming. Additionally, describe the concept of operator precedence and provide examples of how it affects the evaluation of boolean expressions. Discuss the difference between short-circuit evaluation and normal evaluation in boolean expressions and demonstrate their usage in code

Labels — only assistant turns visible (first 400 chars):
 [MASKED][MASKED][MASKED][MASKED][MASKED][MASKED][MASKED][MASKED][MASKED][MASKED][MASKED][MASKED][MASKED][MASKED][MASKED][MASKED][MASKED][MASKED][MASKED][MASKED][MASKED][MASKED][MASKED][MASKED][MASKED][MASKED][MASKED][MASKED][MASKED][MASKED][MASKED][MASKED][MASKED][MASKED][MASKED][MASKED][MASKED][MASKED][MASKED][MASKED][MASKED][MASKED][MASKED][MASKED][MASKED][MASKED][MASKED][MASKED][MASKED][MASKED]


In [8]:
# Log GPU memory before training
gpu_props = torch.cuda.get_device_properties(0)
mem_before = round(torch.cuda.max_memory_reserved() / 1024**3, 2)
total_mem  = round(gpu_props.total_memory       / 1024**3, 2)
print(f"GPU: {gpu_props.name} | Total VRAM: {total_mem} GB | Reserved before training: {mem_before} GB")

GPU: Tesla T4 | Total VRAM: 14.56 GB | Reserved before training: 3.95 GB


In [9]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 99,906 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 33,030,144 of 4,055,498,240 (0.81% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,1.150400
2,1.207000
3,1.235600
4,1.210600
5,1.480300
6,0.867400
7,1.179300
8,0.721300
9,1.069500
10,0.644700


In [10]:
# Training summary
mem_after = round(torch.cuda.max_memory_reserved() / 1024**3, 2)
print(f"Training time : {round(trainer_stats.metrics['train_runtime'] / 60, 2)} min")
print(f"Peak VRAM used: {mem_after} GB  ({round(mem_after/total_mem*100, 1)}% of total)")
print(f"VRAM for LoRA : {round(mem_after - mem_before, 2)} GB")

Training time : 11.08 min
Peak VRAM used: 10.02 GB  (68.8% of total)
VRAM for LoRA : 6.07 GB


## 7. Post-training inference — comparing before vs after

Now we run the **same question** through the trained model using the ChatML template.  
The model should now respond as an assistant rather than completing raw text.

In [11]:
from transformers import TextStreamer

# Use the same question as the baseline, now wrapped in ChatML
messages = [{"role": "user", "content": "What is the capital of France?"}]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,  # True at inference — tells model to produce assistant turn
)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

print("=== FINE-TUNED MODEL (after SFT) ===")
out_ids = model.generate(
    **inputs,
    max_new_tokens = 200,
    temperature    = 0.7,   # Qwen3 recommended instruct settings
    top_p          = 0.8,
    top_k          = 20,
    do_sample      = True,
    streamer       = TextStreamer(tokenizer, skip_prompt=True),
)

after_response = tokenizer.decode(out_ids[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

=== FINE-TUNED MODEL (after SFT) ===
The capital of France is Paris.<|im_end|>


## 8. Side-by-side comparison

In [12]:
print("PROMPT:", raw_prompt)
print()
print("━" * 60)
print("BEFORE SFT (base model, raw text completion):")
print("━" * 60)
print(before_response)
print()
print("━" * 60)
print("AFTER SFT (LoRA fine-tuned, ChatML instruct format):")
print("━" * 60)
print(after_response)

PROMPT: What is the capital of France? Answer:

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
BEFORE SFT (base model, raw text completion):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Paris. But why is Paris the capital of France? Because it is the largest city in France and the political, economic, and cultural center of the country. So the answer is Paris. Now, let's think about the question: What is the capital of Germany? The answer is Berlin. But why is Berlin the capital of Germany? Because it is the largest city in Germany and the political,

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
AFTER SFT (LoRA fine-tuned, ChatML instruct format):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
The capital of France is Paris.


## 9. Try your own prompts

Swap in any question to explore what the fine-tuned model has learned.

In [35]:
def chat(user_message, max_new_tokens=300):
    """Helper: format and generate a single-turn reply from the fine-tuned model."""
    prompt = tokenizer.apply_chat_template(
        [{"role": "user", "content": user_message}],
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.7, top_p=0.8, top_k=20, do_sample=True,
        streamer=TextStreamer(tokenizer, skip_prompt=True),
    )

chat("Explain what Artificial Intelligence in simple terms with some real world examples")

Artificial Intelligence (AI) is a branch of computer science that focuses on creating machines and software that can perform tasks that typically require human intelligence, such as learning, reasoning, problem-solving, perception, and language understanding. AI systems can be trained on large amounts of data to recognize patterns and make decisions without being explicitly programmed to do so.

Real-world examples of AI include:

1. Virtual assistants like Siri, Alexa, and Google Assistant, which can answer questions, set reminders, and control smart home devices.
2. Self-driving cars, which use AI to navigate and avoid obstacles.
3. Recommendation systems like Netflix and Amazon, which use AI to suggest movies and products based on a user's preferences.
4. Chatbots, which use AI to provide customer service and support.
5. Medical diagnosis systems, which use AI to analyze medical images and detect diseases.

Overall, AI has the potential to revolutionize many industries by improving 

## 10. Save the LoRA adapters

LoRA adapters are tiny (a few MB) and can be loaded on top of the frozen base model at any time.  
To deploy, you can either keep them separate or merge them into the base weights.

In [14]:
# Save adapters locally (only the delta weights, not the full model)
model.save_pretrained("qwen3_sft_lora")
tokenizer.save_pretrained("qwen3_sft_lora")

# To push to Hugging Face Hub, uncomment:
# model.push_to_hub("your_username/qwen3_sft_lora", token="YOUR_HF_TOKEN")
# tokenizer.push_to_hub("your_username/qwen3_sft_lora", token="YOUR_HF_TOKEN")

('qwen3_sft_lora/tokenizer_config.json',
 'qwen3_sft_lora/special_tokens_map.json',
 'qwen3_sft_lora/chat_template.jinja',
 'qwen3_sft_lora/vocab.json',
 'qwen3_sft_lora/merges.txt',
 'qwen3_sft_lora/added_tokens.json',
 'qwen3_sft_lora/tokenizer.json')

In [15]:
# Optional: merge adapters into base and save as full float16 model (needed for vLLM / llama.cpp)
# model.save_pretrained_merged("qwen3_sft_merged", tokenizer, save_method="merged_16bit")